# Phase 0 & Phase 1 — Data + structural checks

**Pre-modeling gate:** Run this notebook before any modeling. It covers:

- **Phase 0:** Verify downloaded files and basic counts.
- **Phase 1:** Schema / ID mapping / evaluation-zone structure / missingness.

Requirements: run from repository root (so `import src` resolves). Download data first:

```bash
python scripts/download_data.py
```


## Phase 0 — Get the data

1. Run `python scripts/download_data.py` (needs `~/.kaggle/kaggle.json`).
2. Re-run the code cells below to confirm `train/`, `test/`, and `sample_submission.csv` exist.


In [ ]:
import sys
from pathlib import Path

# Repo root: cwd or first ancestor containing src/
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    for p in ROOT.parents:
        if (p / "src").is_dir():
            ROOT = p
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.io.loaders import (
    discover_wells,
    load_horizontal_well,
    load_sample_submission,
    get_test_column_names,
)
from src.utils.paths import get_data_dir, get_sample_submission_path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
print("ROOT =", ROOT)


In [ ]:
train_dir = get_data_dir("train")
test_dir = get_data_dir("test")
sample_path = get_sample_submission_path()

for label, p in [("train", train_dir), ("test", test_dir), ("sample_submission", sample_path)]:
    ok = p.exists()
    status = "OK" if ok else "MISSING"
    print(f"{label}: {p}  {status}")
    if not ok:
        raise FileNotFoundError(f"Expected {p}. Run: python scripts/download_data.py")

train_wells = discover_wells(train_dir)
test_wells = discover_wells(test_dir)
sub = load_sample_submission(sample_path)

print(f"\nTrain wells: {len(train_wells)}")
print(f"Test wells: {len(test_wells)}")
print(f"sample_submission rows: {len(sub)}  (eval-zone rows total)")


### Phase 0 spot-check (schema)

Open one horizontal file and one typewell manually if you want; columns should match `context/DATASETS.md`.


In [ ]:
example_train = train_wells[0]
hw = load_horizontal_well(train_dir / f"{example_train}__horizontal_well.csv")
tw_path = train_dir / f"{example_train}__typewell.csv"
tw = pd.read_csv(tw_path) if tw_path.exists() else None
print("Horizontal columns:", list(hw.columns))
print("Typewell columns:", list(tw.columns) if tw is not None else "(missing)")
hw.head()


---

## Phase 1 — Structural / schema checks

### 1A — Verify submission ID ↔ CSV row index (Assumption A1)

`id` = `{WELLNAME}_{row_index}`. Confirm every submission row index exists on the horizontal-well DataFrame index (default integer index from CSV).


In [ ]:
sub = load_sample_submission(get_sample_submission_path())

failures = []
for well in test_wells:
    path = test_dir / f"{well}__horizontal_well.csv"
    df = load_horizontal_well(path)
    ids_in_sub = sub.loc[sub["wellname"] == well, "row_idx"].values
    missing = [idx for idx in ids_in_sub if idx not in df.index]
    if missing:
        failures.append((well, missing[:5], len(missing)))

if failures:
    raise AssertionError(f"ID/index mismatches (well, sample_missing): {failures[:3]}...")
print("1A OK: all submission row_idx values exist in each test well index.")


### 1B — Formation columns in test (Decision D002 / Assumption A3)

Log result; update `context/DECISIONS.md` if any train-only formation columns appear in test files.


In [ ]:
train_only = {"ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"}
test_cols = get_test_column_names(test_dir)
present_in_test = sorted(train_only & test_cols)
print("Columns train-only per docs that appear in TEST headers:", present_in_test if present_in_test else "(none)")
print("→ If non-empty, revise D002 and consider those columns as features.")


### 1C — Evaluation-zone structure per training well

Counts known vs NaN `TVT_input` rows and checks whether NaN rows form a single contiguous index block.


In [ ]:
rows = []
for well in sorted(train_wells):
    df = load_horizontal_well(train_dir / f"{well}__horizontal_well.csv")
    n = len(df)
    known = df["TVT_input"].notna()
    eval_mask = df["TVT_input"].isna()
    n_known = int(known.sum())
    n_eval = int(eval_mask.sum())
    idx_nan = sorted(df.index[eval_mask].tolist())
    gaps = 0
    if len(idx_nan) > 1:
        gaps = sum(1 for a, b in zip(idx_nan, idx_nan[1:]) if b != a + 1)
    contiguous = len(idx_nan) <= 1 or gaps == 0
    md_known = df.loc[known, "MD"]
    md_eval = df.loc[eval_mask, "MD"]
    rows.append(
        {
            "well": well,
            "rows": n,
            "n_known": n_known,
            "n_eval": n_eval,
            "eval_frac": n_eval / n if n else 0,
            "md_known_min": md_known.min() if n_known else float("nan"),
            "md_known_max": md_known.max() if n_known else float("nan"),
            "md_eval_min": md_eval.min() if n_eval else float("nan"),
            "md_eval_max": md_eval.max() if n_eval else float("nan"),
            "nan_index_contiguous": contiguous,
            "nan_index_gaps": gaps,
        }
    )

zone_df = pd.DataFrame(rows).sort_values("well")
zone_df


In [ ]:
# Summary: is eval zone usually trailing in MD?
# Uses md_known_max <= md_eval_min as a simple trailing heuristic (relaxed tolerance for float noise).
same_max_md = (zone_df["md_known_max"] <= zone_df["md_eval_min"] + 1e-6).mean()
print(f"Fraction of wells where max(MD_known) <= min(MD_eval): {same_max_md:.3f}")
print("nan_index_contiguous value_counts:")
print(zone_df["nan_index_contiguous"].value_counts())


### 1D — sample_submission rows vs TVT_input NaN rows (test)

For each test well, submission row indices must equal the set of rows where `TVT_input` is NaN.


In [ ]:
sub = load_sample_submission(get_sample_submission_path())
mismatches = []
for well in test_wells:
    df = load_horizontal_well(test_dir / f"{well}__horizontal_well.csv")
    eval_rows = set(df.loc[df["TVT_input"].isna()].index)
    sub_rows = set(sub.loc[sub["wellname"] == well, "row_idx"].astype(int))
    if eval_rows != sub_rows:
        only_eval = sorted(eval_rows - sub_rows)
        only_sub = sorted(sub_rows - eval_rows)
        mismatches.append((well, only_eval[:10], only_sub[:10]))

if mismatches:
    raise AssertionError(f"Eval/submission mismatches: {mismatches[:3]}")
print("1D OK: submission ids match TVT_input NaN rows for all test wells.")


### 1E — Column missingness audit (horizontal well CSV)

Overall vs known-zone vs eval-zone for each column.


In [ ]:
def missingness_report(df: pd.DataFrame, known_mask: pd.Series, eval_mask: pd.Series) -> pd.DataFrame:
    out = []
    for col in df.columns:
        s = df[col]
        out.append(
            {
                "column": col,
                "pct_missing_all": float(s.isna().mean()),
                "pct_missing_known": float(s.loc[known_mask].isna().mean()) if known_mask.any() else float("nan"),
                "pct_missing_eval": float(s.loc[eval_mask].isna().mean()) if eval_mask.any() else float("nan"),
            }
        )
    return pd.DataFrame(out).sort_values("pct_missing_all", ascending=False)


total_rows = 0
agg = None
for well in train_wells:
    df = load_horizontal_well(train_dir / f"{well}__horizontal_well.csv")
    known = df["TVT_input"].notna()
    ev = df["TVT_input"].isna()
    rpt = missingness_report(df, known, ev).set_index("column")
    w = len(df)
    total_rows += w
    agg = rpt * w if agg is None else agg.add(rpt * w, fill_value=0)

miss_summary = (agg / total_rows).reset_index()
miss_summary = miss_summary.sort_values("pct_missing_all", ascending=False)
miss_summary.head(20)


---

### Next steps

- Log verification outcomes in `context/ASSUMPTIONS.md` and `context/DECISIONS.md` when you have results.
- Proceed to **Phase 2** notebook for geology / GR / typewell plots after Phase 1 passes.
